# Plant Instance Wirings — Notebook

End-to-end walkthrough of the two plant-instance retrieval wirings:

- **Plant→Image** (`emb_plant2image.json`) — given a per-plant crop instance, retrieve full field images that contain the same or a similar plant.
- **Plant→Plant** (`emb_plant2plant.json`) — retrieve other instances of the same species using `instance_labels`.

Both wirings convert to the same `MetadataGroup` representation used by Image→Image, so all KPIs are available including graded `knn_metadata_ndcg`.

Run all cells top-to-bottom; all figures are interactive (Plotly).

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from precisionai.agrieval.emb.services.evaluate import (
    run_plant2image_eval,
    run_plant2plant_eval,
)
from precisionai.agrieval.emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_tsne,
    print_result,
)

---
## Part 1 — Plant→Image

`emb_plant2image.json` contains:
- **4 corn parent images** across two L2 clusters (A1, A2), each with **4 instance crops** — 20 corn embeddings.
- **4 soybean parent images** across two L2 clusters (B1, B2), each with **4 instance crops** — 20 soybean embeddings.
- **40 embeddings total** (8 parents + 32 instance crops) across both classes.

Parent images are stored under `images/<cluster>/` and instance crops under `instances/<cluster>/`.  
The `instance_to_image` mapping declares which crops belong to which parent.

Ground truth:
- Instance → its parent is grade-3 (explicit positive).
- Parent → all its instances are grade-3.
- Items sharing the same crop class (A or B) across different parent groups are grade-1.

**Similarity design** — instances of the same parent image have cosine ≈ 0.97–0.99 to their parent (very close, not identical). Parent images of the same class are 0.80–0.95 apart. Cross-class (A vs B) is near zero.

### Load Data

In [2]:
with open("emb_plant2image.json") as f:
    payload = json.load(f)

embeddings = payload["embeddings"]
instance_to_image = payload["instance_to_image"]

print(f"Total embeddings : {len(embeddings)}")
print(f"Parent images    : {len(instance_to_image)}")
print(f"Instance crops   : {sum(len(v) for v in instance_to_image.values())}")
print(f"Embedding dim    : {len(next(iter(embeddings.values())))}")
print()
print("Sample parent->instances mapping:")
for parent, insts in list(instance_to_image.items())[:2]:
    print(f"  {parent}")
    for i in insts:
        print(f"    -> {i}")

Total embeddings : 40
Parent images    : 8
Instance crops   : 32
Embedding dim    : 32

Sample parent->instances mapping:
  images/A1/220622-225403-corn-HB-25000SBC-54e94639.png
    -> instances/A1/220622-225403-corn-HB-25000SBC-54e94639-1.png
    -> instances/A1/220622-225403-corn-HB-25000SBC-54e94639-2.png
    -> instances/A1/220622-225403-corn-HB-25000SBC-54e94639-3.png
    -> instances/A1/220622-225403-corn-HB-25000SBC-54e94639-4.png
  images/A1/220622-091105-corn-HB-25000SBC-8d6b86d0.png
    -> instances/A1/220622-091105-corn-HB-25000SBC-8d6b86d0-1.png
    -> instances/A1/220622-091105-corn-HB-25000SBC-8d6b86d0-2.png
    -> instances/A1/220622-091105-corn-HB-25000SBC-8d6b86d0-3.png
    -> instances/A1/220622-091105-corn-HB-25000SBC-8d6b86d0-4.png


### Run Evaluation

In [3]:
result_p2i = run_plant2image_eval(
    embeddings=embeddings,
    instance_to_image=instance_to_image,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
)
print_result(result_p2i)

n_items      : 40
embedding_dim: 32
classes      : ['A', 'B']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.3219  std=0.4228  (p05=-0.1910  p50=0.1043  p95=0.9576)
  centroid cosine    : mean=0.5821  std=0.0503  norm=0.5821
  intra/inter gap    : 0.8171  (intra=0.7409  inter=-0.0761)
  effective_rank     : 2.38  (ratio=0.0743  dim=32)
  uniformity         : -1.5777
  alignment          : 0.0929

  hubness@5         : mean=5.0000  std=2.0372  p95=10.0000
  hubness@10        : mean=10.0000  std=4.0620  p95=16.1500
  knn_radius@5         : mean=0.7692  std=0.0587  p05=0.6982  p95=0.8840
  knn_radius@10        : mean=0.7065  std=0.0464  p05=0.6583  p95=0.7922
  mean_top_k_sim@5         : mean=0.9167  std=0.0164  p05=0.8957  p95=0.9447
  mean_top_k_sim@10        : mean=0.8243  std=0.0304  p05=0.7867  p95=0.8836
  outlier_score@5         : mean=0.0833  std=0.0164  p95=0.1043
  outlier_score@10        : mean=0.17

### Interpreting the Plant→Image KPIs

| KPI | What to look for |
|---|---|
| `knn_metadata_precision@k` | Are the top-k results explicit positives (parent/instances from the same group)? |
| `knn_metadata_ndcg@k` | Is the full grade-3 set (parent + instances) ranked above grade-1 (same class, different group)? |
| `knn_label_purity@k` | Fraction of neighbours sharing the same crop class (A or B). |
| `alignment` | Mean ‖u−v‖² across explicit positive pairs — lower means parent and instances are closer together. |

With the tight embeddings in this file, you should see **high metadata precision and nDCG** because instances are only 0.97–0.99 cosine away from their parent, while cross-class items are near zero.

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes.

In [4]:
plot_knn_confusion(result_p2i, output_path=None)
plot_cosine_similarity(embeddings, result_p2i, output_path=None)
plot_tsne(embeddings, result_p2i, dimensions=2, output_path=None)

  t-SNE 2D — fitting 40 samples (perplexity=6, iter=1000)...


---
## Part 2 — Plant→Plant

`emb_plant2plant.json` contains 21 instance crops across three classes:
- **`"Crop | Corn"`** — 9 instances (5 from cluster A1, 4 from A2)
- **`"Crop | Soybean"`** — 6 instances (3 from cluster B1, 3 from B2)
- **`"Weed | Weed"`** — 6 instances (3 from cluster C1, 3 from C2)

All instance crops are stored under `instances/<cluster>/`. All instances sharing the same class label are mutual grade-3 positives — useful for measuring class-level retrieval quality.

### Load Data

In [5]:
with open("emb_plant2plant.json") as f:
    payload = json.load(f)

embeddings_p2p = payload["embeddings"]
instance_labels = payload["instance_labels"]

label_counts = Counter(instance_labels.values())
print(f"Total instances   : {len(embeddings_p2p)}")
print(f"Class distribution: {dict(label_counts)}")

Total instances   : 21
Class distribution: {'Crop | Corn': 9, 'Crop | Soybean': 6, 'Weed | Weed': 6}


### Run Evaluation

In [6]:
result_p2p = run_plant2plant_eval(
    embeddings=embeddings_p2p,
    instance_labels=instance_labels,
    k_values=[5, 10],
    sample_pairs=None,
)
print("=== Plant→Plant ===")
print_result(result_p2p)

=== Plant→Plant ===
n_items      : 21
embedding_dim: 32
classes      : ['Crop | Corn', 'Crop | Soybean', 'Weed | Weed']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.1768  std=0.3243  (p05=-0.2276  p50=0.0742  p95=0.7695)
  centroid cosine    : mean=0.4648  std=0.1516  norm=0.4648
  intra/inter gap    : 0.6220  (intra=0.6033  inter=-0.0187)
  effective_rank     : 5.08  (ratio=0.1588  dim=32)
  uniformity         : -2.3993
  alignment          : 0.7933

  hubness@5         : mean=5.0000  std=1.1952  p95=7.0000
  hubness@10        : mean=10.0000  std=4.0356  p95=16.0000
  knn_radius@5         : mean=0.4991  std=0.0871  p05=0.3653  p95=0.6187
  knn_radius@10        : mean=0.0707  std=0.1023  p05=-0.1122  p95=0.2186
  mean_top_k_sim@5         : mean=0.6538  std=0.0568  p05=0.5596  p95=0.7134
  mean_top_k_sim@10        : mean=0.4209  std=0.0858  p05=0.2593  p95=0.5108
  outlier_score@5         : mean=0.3462  st

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes.

In [7]:
plot_knn_confusion(result_p2p, output_path=None)
plot_cosine_similarity(embeddings_p2p, result_p2p, output_path=None)
plot_tsne(embeddings_p2p, result_p2p, dimensions=2, output_path=None)

  t-SNE 2D — fitting 21 samples (perplexity=3, iter=1000)...
